# 运行业务场景评测

先阅读 `../docs/getting_started.md` 并配置 `benchmarks/app_settings.json`。默认仅准备计划，`RUN_TESTS=True` 才会调用产品。新结构请使用新的 campaign 名称；展示用历史报告见 `../docs/reports/`。


In [ ]:
from pathlib import Path
import sys, os, subprocess, json
PROJECT_ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'benchmarks/prepare.py').is_file() and (p/'evaluator/runner.py').is_file()),None)
if PROJECT_ROOT is None: raise RuntimeError('请将本包放到完整项目根目录，再用VS Code打开整个项目')
ENV=dict(os.environ,PYTHONIOENCODING='utf-8',PYTHONUTF8='1')
CAMPAIGN='my_evaluation_v1'
BATCH='smoke'  # 首次8次；之后依次改为'1'到'6'
MODE='rules'   # 本轮固定；若改模式须另起CAMPAIGN
print('项目：',PROJECT_ROOT)
print('先编辑：',PROJECT_ROOT/'benchmarks/app_settings.json')


## 准备本批计划（不调用产品）
先确认上一批已结束。首次smoke=8次；之后批次1自动跳过8次，只剩24次。每批最多32次。

In [ ]:
if 'RUN_JOB' in globals() and RUN_JOB.poll() is None: raise RuntimeError('上一批仍在运行')
prepared=subprocess.run([sys.executable,str(PROJECT_ROOT/'benchmarks/prepare.py'),'--project-root',str(PROJECT_ROOT),'--campaign',CAMPAIGN,'--batch',BATCH,'--mode',MODE],env=ENV,capture_output=True,text=True,encoding='utf-8',errors='replace')
if prepared.returncode: raise RuntimeError(prepared.stderr)
PLAN=json.loads(prepared.stdout);print(json.dumps(PLAN,ensure_ascii=False,indent=2))
CONFIG=Path(PLAN['config_path']) if PLAN.get('config_path') else None
if CONFIG:
    check=subprocess.run([sys.executable,'runner.py','--plan',str(CONFIG)],cwd=PROJECT_ROOT/'evaluator',env=ENV,capture_output=True,text=True,encoding='utf-8',errors='replace')
    print(check.stdout,check.stderr)
    if check.returncode: raise RuntimeError('计划未通过校验')


## 启动本批（会调用产品和测试Agent API）
首批结束后先检查新材料确实生效、回答和来源采集正常，再继续其他批次。

In [ ]:
RUN_TESTS=False
if RUN_TESTS:
    if CONFIG is None: raise RuntimeError('当前批次没有待运行组合')
    if 'RUN_JOB' in globals() and RUN_JOB.poll() is None: raise RuntimeError('已有任务运行中')
    cfg=json.loads(CONFIG.read_text(encoding='utf-8'))
    if cfg.get('campaign_id') != CAMPAIGN or cfg.get('batch_id') != BATCH:
        raise RuntimeError('当前计划与CAMPAIGN/BATCH不一致，请重新运行准备计划单元格')
    print('实际启动计划：', CONFIG)
    print('策略：单个失败后继续，整批结束再补异常')
    if Path(cfg['output_dir']).exists(): raise RuntimeError('此计划已启动过，请重新准备本批计划')
    RUN_LOG=CONFIG.with_suffix('.log')
    with RUN_LOG.open('w',encoding='utf-8') as log:
        RUN_JOB=subprocess.Popen([sys.executable,'runner.py','--config',str(CONFIG)],cwd=PROJECT_ROOT/'evaluator',env=ENV,stdout=log,stderr=subprocess.STDOUT)
    print('本批结果：',cfg['output_dir'])
else: print('未启动；准备好后改RUN_TESTS=True')


In [ ]:
if 'RUN_JOB' in globals():
    print(RUN_LOG.read_text(encoding='utf-8',errors='replace')[-16000:])
    print('返回码（None表示运行中）：',RUN_JOB.poll())
else: print('尚未启动')


## 汇总已执行与已复核结果（不调用API）
依据benchmarks/GOLD_REVIEW.md填写每个结果目录的human_reviews.json后重新运行。尚未复核不会算作通过。

In [ ]:
campaign_dir=PROJECT_ROOT/'evaluation/expanded_campaigns'/CAMPAIGN
result=subprocess.run([sys.executable,str(PROJECT_ROOT/'benchmarks/summarize.py'),str(campaign_dir)],env=ENV,capture_output=True,text=True,encoding='utf-8',errors='replace')
if result.returncode: raise RuntimeError(result.stderr)
print((campaign_dir/'campaign_summary.md').read_text(encoding='utf-8'))

from IPython.display import display, FileLink
for name in ["scenario_recommendations.html", "scenario_recommendations.md", "campaign_matrix.html", "campaign_analysis.md"]:
    target=campaign_dir/name
    print("报告：", target)
    display(FileLink(str(target.relative_to(Path.cwd())) if target.is_relative_to(Path.cwd()) else str(target)))


## 人工对照指标（填写真实数据后才有数值）
先阅读HUMAN_COMPARISON.md。不能拿两次不同产品回答比较裁判准确率；质量实验须使用同一份冻结证据。

In [ ]:
result=subprocess.run([sys.executable,str(PROJECT_ROOT/'benchmarks/compare_metrics.py')],env=ENV,capture_output=True,text=True,encoding='utf-8',errors='replace')
print(result.stdout,result.stderr)
if result.returncode: raise RuntimeError('指标数据无效，请检查填写内容')
